In [ ]:
%cd ../../

In [ ]:
import time
import random
import json
from itertools import combinations, product
import uuid

import pandas as pd
import polars as pl

from src.forecaster import ModelService

In [ ]:
seed = time.time()
random.seed(seed)

In [ ]:
restaurant = 1
schoolyear = "24-25"
date_start = "2025-06-02"

# Load tables

In [ ]:
path = "data/processed/dim_meal_types.xlsx"

dim_meal_types = pl.read_excel(path)
dim_meal_types

In [ ]:
path = "data/processed/dim_meals.parquet"
dim_meals = (
    pl.read_parquet(path)
    # .drop('meal_codes', 'names', 'src')

    # .join(
    #     dim_meal_types.select('meal_type_id', 'meal_type_en'),
    #     left_on='meal_type', right_on='meal_type_id',
    #     how='left'
    # )
)
dim_meals.head()

### Load list of selected meals specific for each restaurant and weekday

File `recommended_meal_list.json` has following structure

```json
{
    "<restaurant_id>": {
        "<weekday_id>": {
            "non-vegan": [<list of meal id>],
            "vegan": [<list of meal id>],
        }
    }
}
```

In [ ]:
path = "data/processed/recommended_meal_list.json"
with open(path) as file:
    meals_by_day_raw = json.load(file)

meals_by_day = {}
for restau, weekdays in meals_by_day_raw.items():
    meals_by_day[int(restau)] = {}
    for weekday, meals in weekdays.items():
        meals_by_day[int(restau)][int(weekday)] = meals

# Craft menus

In [ ]:
NUM_VEGAN_PER_DAY = 2
NUM_MEALS_PER_DAY = 3
MAX_MEAL_OCCURENCES = 2
NUM_FISH_PER_WEEK = 2

NUM_DAY_LEVEL_MENUS = 1_000_000
NUM_WEEK_LEVEL_MENUS = 1000
NUM_MENUS_FINAL = 20

THETA_CO2 = 0.5
THETA_WASTE = 0.04
THETA_KELA = 2
THETA_GLUTEN = 1

ALPHA_POS = 2
ALPHA_CO2 = 1
ALPHA_WASTE = 1
ALPHA_KELA = 2
ALPHA_GLUTEN = 2

## Craft the day-level menus

Day-level menus must satisfy:
- condition (2), (6) and (9)
- containing a variety of meals

***[June 3, 2025]***
- Condition (1.2) and (8) changed from strict to loose

In [ ]:
def craft_day_level_menu(meals_by_day: dict, restaurant: int, weekday: int, n_max: int = 10_000_000) -> list:
    assert restaurant in meals_by_day and weekday in meals_by_day[restaurant]
    non_vegan = meals_by_day[restaurant][weekday]['non-vegan']
    vegan = meals_by_day[restaurant][weekday]['vegan']

    combos = [[x[0], *x[1]] for x in product(non_vegan, combinations(vegan, NUM_VEGAN_PER_DAY))]
    
    return combos[:n_max]

In [ ]:
combos_mon = craft_day_level_menu(meals_by_day, restaurant, 1)
combos_tue = craft_day_level_menu(meals_by_day, restaurant, 2)
combos_wed = craft_day_level_menu(meals_by_day, restaurant, 3)
combos_thu = craft_day_level_menu(meals_by_day, restaurant, 4)
combos_fri = craft_day_level_menu(meals_by_day, restaurant, 5)

# Craft week-level menu

Week-level menus must satisfy:
- condition (1.1), (3)

In [ ]:
meal_type_fish = dim_meal_types.filter(pl.col('meal_type_en') == pl.lit('fish'))['meal_type_id'].head().item()

# Create table containing week menu candidates
menus_week = pl.DataFrame({
    '1': random.choices(combos_mon, k=NUM_DAY_LEVEL_MENUS),
    '2': random.choices(combos_tue, k=NUM_DAY_LEVEL_MENUS),
    '3': random.choices(combos_wed, k=NUM_DAY_LEVEL_MENUS),
    '4': random.choices(combos_thu, k=NUM_DAY_LEVEL_MENUS),
    '5': random.choices(combos_fri, k=NUM_DAY_LEVEL_MENUS),
    'weeklevel_idx': pl.Series([str(uuid.uuid4()) for _ in range(NUM_DAY_LEVEL_MENUS)])
})

menus_week.head()

In [ ]:
ids_valid_week_menu = (
    menus_week
    .select(
        'weeklevel_idx',
        pl.concat_list(["1", "2", "3", "4", "5"]).alias('meal')
    )
    .explode('meal')


    # For each week menu, find the max occurence of meals in the week menu
    .with_columns(
        pl.len().over('weeklevel_idx', 'meal').alias('count_occurence'),
    )
    .with_columns(
        pl.col('count_occurence').max().over('weeklevel_idx').alias('count_max_occurence')
    )

    # Remove week menu candidates not satisfying (3)
    .filter(pl.col('count_max_occurence') <= MAX_MEAL_OCCURENCES)
    


    # Add meal type info and count no. fish meals of each week menu candidate
    .join(dim_meals.select('id', 'meal_type'), left_on='meal', right_on='id', how='left')
    .with_columns(
        (pl.col('meal_type') == meal_type_fish).cast(pl.Int32).alias('is_fish')
    )
    .with_columns(
        pl.col('is_fish').sum().over('weeklevel_idx').alias('count_fish')
    )

    # Remove week menu candidates not satisfying (1.1)
    .filter(pl.col('count_fish') >= NUM_FISH_PER_WEEK)

    .select('weeklevel_idx')
    .unique()
)

menus_week = (
    menus_week
    .join(ids_valid_week_menu, on='weeklevel_idx', how='inner')
    .sample(NUM_WEEK_LEVEL_MENUS)

    .melt(
        'weeklevel_idx',
        value_vars=["1", "2", "3", "4", "5"],
        variable_name="weekday",
        value_name="meal"
    )

    .with_columns(pl.col('weekday').cast(pl.Int32))
)
menus_week.head()

# Add meal-specific info and date-specific needed for calculating score

Following info will be added:
- `whole_pos` (forecasted)
- `whole_waste` (forecasted)
- meal's CO2
- meal's POS (forecasted)
- meal's gluten
- meal's kela

In [ ]:
model = ModelService()

In [ ]:
weekday2date = pl.DataFrame({
    'weekday': [1, 2, 3, 4, 5],
    'date': pl.Series(pd.date_range(date_start, periods=5)).dt.date()
})
weekday2date.head()

In [ ]:
last_date = weekday2date['date'].max().strftime(r"%Y-%m-%d")
pos_restaurant = (
    pl.from_dataframe(model.forecast_pos_restaurant(restaurant, last_date))
    .select(
        pl.col('date').dt.date(),
        pl.col('forecasted').alias('whole_pos')
    )
)
waste_restaurant = (
    pl
    .from_dataframe(model.forecast_waste_restaurant(restaurant, last_date))
    .select(
        pl.col('date').dt.date(),
        pl.col('forecasted').alias('whole_waste')
    )
)

In [ ]:
s_gluten = "gluten_free"
s_kela = "kela"

menus_week = (
    menus_week
    .explode('meal')

    
    .join(weekday2date, on='weekday')
    .join(dim_meals.select('id', 'meal_type'), left_on='meal', right_on='id', how='left')
    .with_columns(pl.col('date').dt.strftime(r"%Y-%m-%d").alias('date_str'))

    # Forecast POS for each meal
    .with_columns(
        pl.struct('meal', 'date_str', 'meal_type')
        .map_elements(
            lambda r: model.forecast_pos_per_meal(restaurant, r['meal'], r['date_str'], r['meal_type']),
            return_dtype=pl.Float32
        ).alias('pos')
    )
    .drop('date_str')

    # Add forecasted restaurant's waste and pos
    .join(pos_restaurant, on='date', how='left')
    .join(waste_restaurant, on='date', how='left')


    # Add CO2, gluten-free and kela for each meal
    .join(
        dim_meals.select(
            'id', 'co2',
            pl.col('attributes').list.contains(s_gluten).alias('is_gluten').cast(pl.Int32),
            pl.col('attributes').list.contains(s_kela).alias('is_kela').cast(pl.Int32),
        ),
        left_on='meal', right_on='id', how='left'
    )
)


menus_week.head()

# Calculate fitness value

In [ ]:
menus_week = (
    menus_week

    # Calculate score for each date
    .group_by('weeklevel_idx', 'weekday')
    .agg(
        pl.concat_list(pl.struct('meal', 'pos')).flatten().alias('meals_planned'),

        pl.col('pos').sum().alias('sum_pos'),
        (pl.col('pos') * pl.col('co2')).sum().alias('sum_co2_pos'),
        pl.col('is_gluten').sum().alias('sum_gluten'),
        pl.col('is_kela').sum().alias('sum_kela'),

        pl.col('whole_pos').first(),
        pl.col('whole_waste').first(),
    )

    .with_columns(
        (
            ALPHA_POS * (pl.col('sum_pos') / pl.col('whole_pos') - 1).abs()
            + ALPHA_CO2 * (pl.col('sum_co2_pos') / pl.col('sum_pos') / THETA_CO2)
            + ALPHA_WASTE * (pl.col('whole_waste') / pl.col('sum_pos') / THETA_WASTE)
            + ALPHA_GLUTEN * (1 - pl.col('sum_gluten') / THETA_GLUTEN).clip(0)
            + ALPHA_KELA * (1 - pl.col('sum_kela') / THETA_GLUTEN).clip(0)
        ).alias('score_day')
    )

    # Calculate score for entire week
    .with_columns(
        pl.col('score_day').sum().over('weeklevel_idx').alias('score_week')
    )
    .with_columns(
        pl.col('score_week').rank('dense', descending=False).over(None).alias('rank')
    )
    .filter(pl.col('rank') <= NUM_MENUS_FINAL)
    .sort('rank')


    # Keep columns as data model
    .join(weekday2date, on='weekday', how='left')
    .select(
        pl.concat_str(
            [
                pl.col('weeklevel_idx'),
                pl.col('date').dt.strftime(r"%Y-%m-%d"), 
                pl.lit(restaurant)
            ],
            separator='|'
        ).alias('id'),
        'weeklevel_idx',
        'date',
        'whole_waste',
        'whole_pos',
        'score_week',
        'meals_planned'
    )
)

menus_week.head()

In [ ]:
dim_meals_planned = (
    menus_week
    .select(pl.col('id').alias('menu_id'), 'meals_planned')
    .explode('meals_planned')
    .unnest('meals_planned')
)

dim_meals_planned.head()

# Test the crafted menus

Check condition: (1.1), (1.2), (2), (6)

In [ ]:
meal_type_vegan = dim_meal_types.filter(pl.col('meal_type_en').is_in(['vegan', 'vegetarian']))['meal_type_id']

weeklevel_idx = "34f4ad10-35c8-4cfb-a7f1-4f18c6477f3b"

(
    menus_week
    # .filter(pl.col('weeklevel_idx') == pl.lit(weeklevel_idx))

    .select('weeklevel_idx', 'date', 'meals_planned')
    .explode('meals_planned')
    .unnest('meals_planned')

    .join(
        dim_meals.select(
            'id',
            (pl.col('meal_type') == meal_type_fish).cast(pl.Int32).alias('is_fish'),
            (pl.col('meal_type').is_in(meal_type_vegan)).cast(pl.Int32).alias('is_vegan'),
            pl.col('attributes').list.contains(s_gluten).alias('is_gluten').cast(pl.Int32),
            pl.col('attributes').list.contains(s_kela).alias('is_kela').cast(pl.Int32),
        ),
        left_on='meal', right_on='id', how='left'
    )
    .group_by('weeklevel_idx', 'date')
    .agg(
        pl.len().alias('no_meals_per_day'),
        pl.col('is_fish').sum(),
        pl.col('is_vegan').sum(),
        pl.col('is_gluten').sum(),
        pl.col('is_kela').sum()
    )
    .group_by('weeklevel_idx')
    .agg(
        pl.col('no_meals_per_day').min(),
        pl.col('is_fish').sum(),
        pl.col('is_vegan').min(),
        pl.col('is_gluten').mean(),
        pl.col('is_kela').mean(),
    )

)

Check condition (3)

In [ ]:
(
    menus_week
    # .filter(pl.col('weeklevel_idx') == pl.lit(weeklevel_idx))

    .select('weeklevel_idx', 'date', 'meals_planned')
    .explode('meals_planned')
    .unnest('meals_planned')

    .group_by('weeklevel_idx', 'meal')
    .len('no_occurrences')
    .group_by('weeklevel_idx')
    .agg(
        pl.col('no_occurrences').max().alias('no_max_occurrences')
    )

    .filter(pl.col('no_max_occurrences') > MAX_MEAL_OCCURENCES)
)